In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

SPLIT_VERSION = 'v2_64_16_20'
RANDOM_STATE = 42
engineered_path = DATA_DIR / 'engineered_features.pkl'
processed_path = DATA_DIR / 'processed_split.pkl'

if not engineered_path.exists():
    raise FileNotFoundError('Missing data/engineered_features.pkl. Run 02_feature_engineering.ipynb first.')

df = joblib.load(engineered_path)
X = df.drop(columns=['isFraud', 'isFlaggedFraud'])
y = df['isFraud']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=RANDOM_STATE
)

# Chuẩn hóa
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

generated_at_utc = datetime.now(timezone.utc).isoformat()
split_metadata = {
    'split_version': SPLIT_VERSION,
    'generated_at_utc': generated_at_utc,
    'random_state': RANDOM_STATE,
    'train_fraction': len(X_train) / len(X),
    'validation_fraction': len(X_val) / len(X),
    'test_fraction': len(X_test) / len(X),
    'train_rows': len(X_train),
    'validation_rows': len(X_val),
    'test_rows': len(X_test),
    'n_features': X_train.shape[1],
}

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(X_train), len(X_val), len(X_test)],
    'fraction_of_full_data': [len(X_train) / len(X), len(X_val) / len(X), len(X_test) / len(X)],
    'fraud_rows': [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    'fraud_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
    'n_features': [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
    'split_version': SPLIT_VERSION,
    'generated_at_utc': generated_at_utc,
})

# lưu vào processed_split.pkl
joblib.dump({
    'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    'X_train_scaled': X_train_scaled, 'X_val_scaled': X_val_scaled, 'X_test_scaled': X_test_scaled,
    'scaler': scaler, 'split_metadata': split_metadata,
}, processed_path)
split_summary.to_csv(RESULTS_DIR / f'split_summary_{SPLIT_VERSION}.csv', index=False)

assert abs(split_metadata['train_fraction'] - 0.64) < 0.001
assert abs(split_metadata['validation_fraction'] - 0.16) < 0.001
assert abs(split_metadata['test_fraction'] - 0.20) < 0.001
print(split_summary.to_string(index=False, formatters={
    'fraction_of_full_data': '{:.4%}'.format, 'fraud_rate': '{:.4%}'.format
}))


     split    rows fraction_of_full_data  fraud_rows fraud_rate  n_features split_version                 generated_at_utc
     train 4072076              64.0000%        5256    0.1291%          15   v2_64_16_20 2026-08-20T09:38:14.782010+00:00
validation 1018020              16.0000%        1314    0.1291%          15   v2_64_16_20 2026-08-20T09:38:14.782010+00:00
      test 1272524              20.0000%        1643    0.1291%          15   v2_64_16_20 2026-08-20T09:38:14.782010+00:00


In [2]:
print(
    f"Split generated: version={SPLIT_VERSION}, timestamp_utc={generated_at_utc}, "
    f"train={X_train.shape}, val={X_val.shape}, test={X_test.shape}"
)
print(f"Saved split artifact: {processed_path}")


Split generated: version=v2_64_16_20, timestamp_utc=2026-08-20T09:38:14.782010+00:00, train=(4072076, 15), val=(1018020, 15), test=(1272524, 15)
Saved split artifact: d:\GitHub\fraud-detection-thesis\data\processed_split.pkl
